In [1]:
import os
os.chdir('/workspace/3fc5e09e-31ce-431d-afd6-9c9d57edec5f')
print(os.listdir('.'))


['ldelta_zeros_N1455_dps80_partial.npy', 'generate_ldelta.py', 'memory', 'refine_shard_runner.py', 'ldelta_zeros_N3405_approx_dps38.npy', '.kernel_llm_logs_1.txt', '.prompts', '.config', '-PROMPT-v6-DATASET.md']


In [2]:
import numpy as np
a = np.load('ldelta_zeros_N1455_dps80_partial.npy')
b = np.load('ldelta_zeros_N3405_approx_dps38.npy')
print(a.shape, a.dtype, a[0], a[-1])
print(b.shape, b.dtype, b[0], b[-1], b[1454], b[1455])


(1455,) <U81 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290 1095.8905568079281792848599941778506664840443308606502951853687094242569554427745
(3405,) <U51 9.2223793999211025222437671927434781355 2199.5340780429643642638069007845685866 1095.8905568079281792848599941778506665 1096.4504813843971045427131177495825196


In [3]:
import subprocess, sys
# Test cypari2 availability
out = subprocess.run([sys.executable, '-c', 'import cypari2; p=cypari2.Pari(); print(p.version())'], capture_output=True, text=True)
print('STDOUT:', out.stdout)
print('STDERR:', out.stderr)
print('CPUs:', os.cpu_count())


STDOUT: 
STDERR: Traceback (most recent call last):
 File "<string>", line 1, in <module>
 import cypari2; p=cypari2.Pari(); print(p.version())
 ^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'cypari2'

CPUs: 8


In [4]:
import subprocess
r = subprocess.run(['pip', 'install', 'cypari2', 'cysignals'], capture_output=True, text=True, timeout=600)
print('OUT:', r.stdout[-2000:])
print('ERR:', r.stderr[-2000:])


OUT: Collecting cypari2

ERR: 


In [5]:
import subprocess, sys
out = subprocess.run([sys.executable, '-c', 'import cypari2; p=cypari2.Pari(); print(p.default("realprecision"))'], capture_output=True, text=True, timeout=60)
print('STDOUT:', out.stdout); print('STDERR:', out.stderr)


STDOUT: 15

STDERR: 


In [6]:

# Quick smoke test: initialize L(Delta, s) and refine the first zero from approximate to dps=80,
# measuring elapsed time. This gives us an empirical cost baseline.
test_code = r"""
import time, cypari2, sys
pari = cypari2.Pari()
pari.allocatemem(10**9)
pari.default('realprecision', 80)
t0 = time.time()
pari('L = lfunmf(mfinit([1,12],1), mfeigenbasis(mfinit([1,12],1))[1])')
print(f'init: {time.time()-t0:.2f}s', flush=True)

# Refine zero around g0=9.2223794
import numpy as np
b = np.load('/workspace/3fc5e09e-31ce-431d-afd6-9c9d57edec5f/ldelta_zeros_N3405_approx_dps38.npy')
for idx in (0, 1454, 1500, 3000, 3404):
 approx = b[idx]
 t0 = time.time()
 pari(f'g0 = {approx}')
 bracket = None
 for w in (1e-2, 1e-3, 5e-4, 1e-4, 5e-5):
 fa = float(pari(f'lfunhardy(L, g0 - {w})'))
 fb = float(pari(f'lfunhardy(L, g0 + {w})'))
 if fa * fb < 0:
 bracket = w; break
 if bracket is None:
 print(idx, 'NO_BRACKET'); continue
 pari(f'r = solve(t = g0 - {bracket}, g0 + {bracket}, lfunhardy(L, t))')
 refined = str(pari(f'Strprintf("%.80g", r)'))
 print(f'idx={idx} approx_T={float(approx):.2f} refined_in {time.time()-t0:.2f}s -> {refined[:30]}...', flush=True)
"""
with open('/tmp/smoke.py','w') as f:
 f.write(test_code)
import subprocess, sys
r = subprocess.run([sys.executable, '/tmp/smoke.py'], capture_output=True, text=True, timeout=600)
print(r.stdout)
print('ERR:', r.stderr[-2000:])


PARI stack size set to 1000000000 bytes, maximum size set to 1000001536
init: 0.03s
idx=0 approx_T=9.22 refined_in 0.01s -> 9.2223793999211025222437671927...
idx=1454 approx_T=1095.89 refined_in 5.91s -> 1095.8905568079281792848599941...
idx=1500 approx_T=1123.82 refined_in 6.25s -> 1123.8170501091986613951617169...
idx=3000 approx_T=1981.29 refined_in 27.40s -> 1981.2864084664966753791786049...
idx=3404 approx_T=2199.53 refined_in 36.13s -> 2199.5340780429643642638069007...

ERR: 


In [7]:

# Estimate refinement total cost for indices 1455..3404 (1950 zeros) assuming T^1.99 per zero.
# Empirical data:
# idx=1454, T~1095.89 -> 5.91s
# idx=3000, T~1981.29 -> 27.4s
# idx=3404, T~2199.53 -> 36.13s
import numpy as np
b = np.load('ldelta_zeros_N3405_approx_dps38.npy')
ts = np.array([float(x) for x in b[1455:3405]])
# Fit cost ~ a * T^p using two anchor points: (1095.89, 5.91) and (2199.53, 36.13)
# p = log(36.13/5.91)/log(2199.53/1095.89)
p = np.log(36.13/5.91)/np.log(2199.53/1095.89)
a = 5.91 / (1095.89**p)
print(f'fit: cost = {a:.3e} * T^{p:.3f}')
costs = a * ts**p
print(f'total refinement seconds (serial): {costs.sum():.1f}')
print(f'with 8 cores: {costs.sum()/8:.1f}s = {costs.sum()/8/60:.1f} min')
print(f'mean per zero: {costs.mean():.2f}s, max: {costs.max():.2f}s')


fit: cost = 7.448e-08 * T^2.599
total refinement seconds (serial): 36456.2
with 8 cores: 4557.0s = 76.0 min
mean per zero: 18.70s, max: 36.13s


In [8]:

# Cost estimate is concerning. With only 3600s total budget and ~4557s minimum for refinement alone
# (best case 8 cores), we cannot complete the full N=5000 task within budget.
# Also need: search for ~1600 new zeros from T~2200 to T~3033 (at dps=38 ~ slow as well), 
# then refine those (T~2200-3033, even higher cost per zero).
#
# Let me get a better empirical estimate of the search cost and re-evaluate.
# But first, the realistic hard budget here is 3600s wall-clock for this kernel.
# Let me be transparent: we should attempt to maximize progress within the budget,
# document the actual outcome, and produce a checkpoint that can be resumed.
#
# Plan:
# 1) Refine indices 1455..3404 in parallel via 8 subprocess shards (with checkpointing TSVs).
# 2) Reassess remaining budget; if any time left, launch search beyond T=2200.
# 3) Assemble best-effort final list; if we don't reach N=5000, save N=<actual> and report honestly.

# First, write the work plan and prepare per-shard pickle inputs.
import numpy as np, pickle, os

b = np.load('ldelta_zeros_N3405_approx_dps38.npy')
remaining_idx = list(range(1455, 3405)) # 1950 zeros
print('Remaining to refine:', len(remaining_idx))

os.makedirs('refine_shards', exist_ok=True)

# Cost-aware partition: assign zeros to 8 shards so each shard has approximately equal sum of T^2.6
ts = np.array([float(b[i]) for i in remaining_idx])
p = 2.599
costs = ts**p
total = costs.sum()
target = total / 8

# Greedy split into 8 contiguous chunks (preserves locality for PARI cache friendliness)
# Actually, easier: stride/round-robin to even out costs across shards.
shards = [[] for _ in range(8)]
shard_costs = [0.0]*8
# Sort remaining by cost descending; assign each to the shard with the lowest current cost
order = np.argsort(-costs)
for k in order:
 s = int(np.argmin(shard_costs))
 shards[s].append((remaining_idx[k], b[remaining_idx[k]]))
 shard_costs[s] += costs[k]

for i in range(8):
 print(f'shard {i}: n={len(shards[i])}, est_cost={shard_costs[i]:.1f}s')
 with open(f'refine_shards/shard_{i}.pkl', 'wb') as f:
 pickle.dump(shards[i], f)

print('max shard est:', max(shard_costs))


Remaining to refine: 1950
shard 0: n=244, est_cost=61326151501.0s
shard 1: n=244, est_cost=61326330026.4s
shard 2: n=244, est_cost=61326221237.4s
shard 3: n=243, est_cost=61246466768.0s
shard 4: n=244, est_cost=61325979128.9s
shard 5: n=244, est_cost=61326121536.2s
shard 6: n=244, est_cost=61325998453.8s
shard 7: n=243, est_cost=61246461071.9s
max shard est: 61326330026.401505


In [9]:

# Costs above are not normalized — they're raw T^p; the absolute time is governed by the prefactor a=7.45e-8.
# Sanity check: estimated max wall time per shard:
a = 7.448e-8
print('max shard wall (s):', a*max([61326330026.4]))
print('average shard wall (s):', a*sum([61326151501.0])/1) # ~4564s actually no, sum of all shards
total = 7.448e-8 * (61326151501 + 61326330026 + 61326221237 + 61246466768 + 61325979128 + 61326121536 + 61325998453 + 61246461071)
print('serial total expected (s):', total)
print('per-shard expected (s):', total/8)


max shard wall (s): 4567.585060366272
average shard wall (s): 4567.57176379448
serial total expected (s): 36528.6958695456
per-shard expected (s): 4566.0869836932


In [10]:

# Each shard expected ~4566s wall — exceeds the 3600s kernel budget.
# We must be realistic: we cannot complete the full N=5000 task in 3600s on this kernel.
# 
# Strategy: launch all 8 refinement shards now and run for ~3300s (saving 300s for assembly).
# The checkpointing TSV ensures partial progress is preserved.
# Then assemble whatever is complete, report progress honestly.

import subprocess, sys, os, time

here = os.path.abspath('.')
runner = os.path.join(here, 'refine_shard_runner.py')
print('runner exists:', os.path.exists(runner))

# Pre-clean any old TSVs
for i in range(8):
 p = f'refine_shards/refined_{i}.tsv'
 if os.path.exists(p):
 os.remove(p)

procs = []
logs = []
t_start = time.time()
for i in range(8):
 pkl = f'refine_shards/shard_{i}.pkl'
 out_txt = f'refine_shards/refined_{i}.tsv'
 log_path = f'refine_shards/shard_{i}.log'
 lf = open(log_path, 'w')
 cmd = [sys.executable, runner, str(i), pkl, '80', out_txt]
 p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT)
 procs.append(p); logs.append(lf)
 print(f'launched shard {i} pid={p.pid}')

print(f'all 8 shards launched at t=0')


runner exists: True
launched shard 0 pid=188
launched shard 1 pid=189
launched shard 2 pid=190
launched shard 3 pid=191
launched shard 4 pid=192
launched shard 5 pid=193
launched shard 6 pid=194
launched shard 7 pid=195
all 8 shards launched at t=0


In [11]:

import time, os, glob
# Poll progress while shards run
def progress():
 total_done = 0
 for i in range(8):
 p = f'refine_shards/refined_{i}.tsv'
 n = 0
 if os.path.exists(p):
 with open(p) as f:
 n = sum(1 for _ in f)
 total_done += n
 return total_done

time.sleep(120)
print(f't={time.time()-t_start:.0f}s done={progress()}/1950 alive={sum(p.poll() is None for p in procs)}')


t=127s done=16/1950 alive=8


In [12]:

# At 127s, 16 zeros done — pace of ~0.126 zeros/s overall (i.e. ~7.9s per zero).
# To finish 1950 zeros at this pace would need ~1950/0.126 ≈ 15500s of wall — even slower than predicted.
# But cost-aware partitioning means most shards are running on smaller T zeros first or distributed mix.
# Each shard has 244 zeros and an expected 4566s of work, so completing all 244 takes ~4566s.
#
# Let's continue polling at intervals and report progress. We will run until ~3200s elapsed,
# then terminate, assemble what we have, and report honestly.

import time
checkpoints = []
for sleep_s in [180, 300, 300, 300, 300, 300, 300, 300, 300, 300]:
 time.sleep(sleep_s)
 elapsed = time.time() - t_start
 done = progress()
 alive = sum(p.poll() is None for p in procs)
 rate = done / elapsed
 eta = (1950 - done)/rate if rate > 0 else float('inf')
 print(f't={elapsed:.0f}s done={done}/1950 alive={alive} rate={rate:.3f}/s ETA={eta:.0f}s', flush=True)
 checkpoints.append((elapsed, done, alive))
 if alive == 0 or done >= 1950:
 break
 if elapsed > 3100:
 print('time budget exhausted; terminating shards')
 for p in procs:
 if p.poll() is None:
 p.terminate()
 time.sleep(3)
 for p in procs:
 if p.poll() is None:
 p.kill()
 break


TimeoutError: Code execution timed out after 900.0 seconds

In [13]:

# Cell timed out (kernel cell-level timeout of 900s) — but the subprocesses keep running.
# Check progress now.
import time, os
def progress():
 total_done = 0
 per = []
 for i in range(8):
 p = f'refine_shards/refined_{i}.tsv'
 n = 0
 if os.path.exists(p):
 with open(p) as f:
 n = sum(1 for _ in f)
 per.append(n)
 total_done += n
 return total_done, per

# Check process state via /proc
import psutil
alive = 0
pids_alive = []
for p in procs:
 try:
 if p.poll() is None:
 alive += 1
 pids_alive.append(p.pid)
 except: pass

elapsed = time.time() - t_start
done, per_shard = progress()
print(f't={elapsed:.0f}s done={done}/1950 alive={alive}')
print('per-shard:', per_shard)


t=1048s done=139/1950 alive=0
per-shard: [17, 18, 17, 17, 18, 18, 17, 17]


In [14]:

# Bad news: alive=0 — all shards died! The cell timeout killed the kernel's subprocess group.
# (Each cell timeout reset the cell, but the Popen children share the kernel process group.)
# Let me verify and check shard logs.

for i in range(8):
 logp = f'refine_shards/shard_{i}.log'
 if os.path.exists(logp):
 with open(logp) as f:
 content = f.read()
 if content.strip():
 print(f'--- shard {i} log (last 400 chars) ---')
 print(content[-400:])


--- shard 0 log (last 400 chars) ---
, in main
 pari(f'r = solve(t = g0 - {bracket}, g0 + {bracket}, lfunhardy(L, t))')
 ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "cypari2/pari_instance.pyx", line 800, in cypari2.pari_instance.Pari.__call__
 File "cypari2/gen.pyx", line 4784, in cypari2.gen.objtogen
 File "cypari2/convert.pyx", line 545, in cypari2.convert.PyObject_AsGEN
KeyboardInterrupt

--- shard 1 log (last 400 chars) ---
, in main
 pari(f'r = solve(t = g0 - {bracket}, g0 + {bracket}, lfunhardy(L, t))')
 ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "cypari2/pari_instance.pyx", line 800, in cypari2.pari_instance.Pari.__call__
 File "cypari2/gen.pyx", line 4784, in cypari2.gen.objtogen
 File "cypari2/convert.pyx", line 545, in cypari2.convert.PyObject_AsGEN
KeyboardInterrupt

--- shard 2 log (last 400 chars) ---
, in main
 pari(f'r = solve(t = g0 - {bracket}, g0 + {bracket}, lfunhardy(L, t))')
 ~~~~^^^^^^^^^^^^^^^^^

In [15]:

# The cell timeout sent SIGINT to children. Need to launch shards detached (preexec_fn=os.setsid)
# so they survive the kernel cell timeout. Restart with daemonized setup.
#
# Note: we have 139 zeros refined already from the previous attempt. The TSV checkpoints persist.
# Re-launch with proper detachment (start_new_session=True) so they survive kernel interruption.
#
# Also key insight: each shard takes ~7s/zero on average. After 1048s wall, 17-18/244 zeros done
# means we'd need ~244*7 = 1700s per shard to finish all 244. That's exactly the cost we predicted.
# A full refinement of remaining 1950 zeros takes ~4500s wall (one shard's worth of work).

# Re-launch shards detached, using start_new_session=True so SIGINT to kernel does not propagate.
import subprocess, sys, os, time

here = os.path.abspath('.')
runner = os.path.join(here, 'refine_shard_runner.py')

procs2 = []
t_resume = time.time()
for i in range(8):
 pkl = f'refine_shards/shard_{i}.pkl'
 out_txt = f'refine_shards/refined_{i}.tsv'
 log_path = f'refine_shards/shard_{i}.log'
 lf = open(log_path, 'a')
 cmd = [sys.executable, runner, str(i), pkl, '80', out_txt]
 p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True)
 procs2.append(p)
 print(f'resumed shard {i} pid={p.pid}')

print(f'all 8 shards re-launched (detached)')


resumed shard 0 pid=260
resumed shard 1 pid=261
resumed shard 2 pid=262
resumed shard 3 pid=263
resumed shard 4 pid=264
resumed shard 5 pid=265
resumed shard 6 pid=266
resumed shard 7 pid=267
all 8 shards re-launched (detached)


In [16]:

# Now poll in short sleep batches that don't exceed cell timeouts.
import time, os

def progress():
 total = 0; per = []
 for i in range(8):
 p = f'refine_shards/refined_{i}.tsv'
 n = 0
 if os.path.exists(p):
 with open(p) as f:
 n = sum(1 for _ in f)
 per.append(n); total += n
 return total, per

import psutil
def alive_count():
 a = 0
 for p in procs2:
 if p.poll() is None: a += 1
 return a

time.sleep(60)
d, per = progress()
print(f'After +60s on resumed: total={d}, per={per}, alive={alive_count()}')


After +60s on resumed: total=147, per=[18, 19, 18, 18, 19, 19, 18, 18], alive=8


In [17]:

# Good — only 8 new zeros in 60s (1 per shard). The shards correctly skip already-done from checkpoints
# and resume on the next pending zero. But they had to re-init PARI/lfun (~30s startup overhead).
#
# Rate is ~0.13 zeros/sec aggregated. Need ~1800 more zeros at ~0.13/s = ~14000s wall. Impossible.
#
# We must accept: full N=5000 list cannot be produced in the remaining budget.
# We will get as much progress as possible, then assemble partial result.
#
# Strategy: poll in 5x180s waits = 15 minutes. Then assemble whatever we have.
import time
for _ in range(5):
 time.sleep(180)
 d, per = progress()
 print(f't+{int(time.time()-t_resume)}s total_done={d} per={per} alive={alive_count()}', flush=True)


TimeoutError: Code execution timed out after 900.0 seconds

In [18]:

import time, os
def progress():
 total = 0; per = []
 for i in range(8):
 p = f'refine_shards/refined_{i}.tsv'
 n = 0
 if os.path.exists(p):
 with open(p) as f:
 n = sum(1 for _ in f)
 per.append(n); total += n
 return total, per

# Check if detached subprocesses are still alive via psutil
import psutil
pids = [260,261,262,263,264,265,266,267]
alive = []
for pid in pids:
 try:
 proc = psutil.Process(pid)
 if proc.is_running() and proc.status() != psutil.STATUS_ZOMBIE:
 alive.append(pid)
 except psutil.NoSuchProcess:
 pass
d, per = progress()
print(f'progress now: total={d} per={per} alive_pids={alive} elapsed_resume={time.time()-t_resume:.0f}s')


progress now: total=283 per=[35, 37, 34, 35, 36, 36, 34, 36] alive_pids=[260, 261, 262, 263, 264, 265, 266, 267] elapsed_resume=986s


In [19]:

# Detached shards are alive and progressing. 283 total refined so far (139 from first run + 144 new).
# Rate since resume: 144/986 = 0.146/s aggregate. To finish remaining 1950-283=1667 zeros: ~11400s.
# We have ~3600s total cell budget. We've already consumed ~2400s of it (from earlier cells + first launch + this poll).
# 
# Let me track remaining budget conservatively. Continue polling in shorter chunks now.
import time
for _ in range(2):
 time.sleep(300)
 d, per = progress()
 pids_alive = []
 for pid in [260,261,262,263,264,265,266,267]:
 try:
 import psutil
 if psutil.Process(pid).is_running(): pids_alive.append(pid)
 except: pass
 print(f't+{int(time.time()-t_resume)}s total_done={d}/1950 alive={len(pids_alive)}', flush=True)


t+1294s total_done=334/1950 alive=8


t+1594s total_done=386/1950 alive=8


In [20]:

# Budget warning: 663s remaining. Stop polling and assemble what we have now.
# We will:
# 1. Read all TSV checkpoints
# 2. Merge with the 1455 dps=80 zeros already on disk
# 3. Sort + dedup + save partial result
# 4. Validate gamma_1 against LMFDB
# Then submit honest report.

import os, numpy as np
from collections import defaultdict

# Collect all refined zeros from TSV shards
refined = {} # orig_idx -> refined str
errors = []
for i in range(8):
 p = f'refine_shards/refined_{i}.tsv'
 if not os.path.exists(p):
 continue
 with open(p) as f:
 for line in f:
 line = line.rstrip('\n')
 if not line: continue
 parts = line.split('\t')
 if len(parts) < 3: continue
 try:
 oi = int(parts[0])
 except: 
 continue
 val = parts[1]
 if val.startswith('ERROR'):
 errors.append((oi, val))
 continue
 refined[oi] = val

print(f'refined from TSVs: {len(refined)} (errors: {len(errors)})')
if errors:
 print('first errors:', errors[:3])

# Load existing 1455 dps=80 zeros
existing = np.load('ldelta_zeros_N1455_dps80_partial.npy')
print(f'existing dps=80: {len(existing)}')

# Approximate list
b38 = np.load('ldelta_zeros_N3405_approx_dps38.npy')

# Combine: indices 0..1454 from existing; indices >=1455 from refined where available
combined = []
for i in range(1455):
 combined.append(existing[i])
for i in sorted(refined.keys()):
 combined.append(refined[i])

print(f'combined dps=80 count: {len(combined)}')

# Convert to floats for sort + dedup
pairs = []
for s in combined:
 try:
 v = float(s)
 pairs.append((v, s))
 except:
 pass
pairs.sort(key=lambda x: x[0])

dedup = []
prev = None
TOL = 1e-9
for v, s in pairs:
 if prev is not None and abs(v-prev) < TOL: continue
 dedup.append(s); prev = v

print(f'after dedup: {len(dedup)}')


refined from TSVs: 388 (errors: 0)
existing dps=80: 1455
combined dps=80 count: 1843
after dedup: 1843


In [21]:

# Validate gamma_1 against LMFDB reference
GAMMA1_REF = "9.2223793999211025222437671927434781355287706224320092899981864280019212953707290061860977049867618"
g1 = dedup[0]
nmatch = 0
for a,b_ in zip(g1, GAMMA1_REF):
 if a == b_: nmatch += 1
 else: break
print(f'gamma_1 = {g1}')
print(f'matching characters with LMFDB ref: {nmatch}')
# Save as <U81
arr = np.array(dedup, dtype='<U81')
# Save partial extended list (this is the new best-effort artifact)
np.save('ldelta_zeros_N1843_dps80_partial.npy', arr)
print(f'Saved {arr.shape[0]} zeros to ldelta_zeros_N1843_dps80_partial.npy (dtype={arr.dtype})')
print(f'First zero: {arr[0]}')
print(f'Last zero: {arr[-1]}')

# Check contiguity: do we have the first N consecutive zeros (no gaps)?
import numpy as np
b38 = np.load('ldelta_zeros_N3405_approx_dps38.npy')
contiguous_indices = set(range(1455))
contiguous_indices.update(refined.keys())
# Find longest contiguous prefix from 0
n_contig = 0
for i in range(3405):
 if i in contiguous_indices:
 n_contig = i+1
 else:
 break
print(f'Largest contiguous prefix at dps=80: {n_contig} zeros')


gamma_1 = 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290
matching characters with LMFDB ref: 81
Saved 1843 zeros to ldelta_zeros_N1843_dps80_partial.npy (dtype=<U81)
First zero: 9.2223793999211025222437671927434781355287706224320092899981864280019212953707290
Last zero: 2199.5340780429643642638069007845685866234965839396509440465393336940850488826679
Largest contiguous prefix at dps=80: 1455 zeros


In [22]:

# The 388 newly-refined zeros are scattered across the [1455, 3404] range (cost-aware distribution),
# so the contiguous prefix at dps=80 remains 1455. The N=5000 dps=80 target is far from complete.
#
# Subprocesses are still running — leave them running until we exit so they continue checkpointing.
# Save a "save-final-state" snapshot that lists which orig_indices are refined.

# Save the index-keyed refined map and stats
import pickle
with open('ldelta_refined_partial_state.pkl','wb') as f:
 pickle.dump({
 'refined_orig_indices_above_1454': sorted(refined.keys()),
 'count_above_1454': len(refined),
 'contiguous_dps80_prefix': 1455,
 'errors': errors,
 }, f)

# Summarize how much we accomplished
elapsed_total = (1594 + 1048) # rough total wall-time consumed across both refinement attempts
new_zeros = len(refined)
rate = new_zeros / elapsed_total
print(f'Refined {new_zeros} new dps=80 zeros in ~{elapsed_total}s wall ({rate:.3f}/s aggregate)')
print(f'Time to finish remaining 1562 zeros at current pace: {1562/rate:.0f}s = {1562/rate/3600:.1f} hours')

# Final figure: progress + cost model
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

ts = np.array([float(b38[i]) for i in range(3405)])
refined_idx = sorted(list(range(1455)) + sorted(refined.keys()))
refined_T = np.array([float(b38[i]) for i in refined_idx])

# Per-zero elapsed times from TSVs
per_zero_times = []
for i in range(8):
 p = f'refine_shards/refined_{i}.tsv'
 if not os.path.exists(p): continue
 with open(p) as f:
 for line in f:
 parts = line.rstrip().split('\t')
 if len(parts) >= 3 and not parts[1].startswith('ERROR'):
 try:
 oi = int(parts[0]); el = float(parts[2])
 per_zero_times.append((float(b38[oi]), el))
 except: pass

per_zero_times.sort()
T_vals = np.array([t for t,_ in per_zero_times])
e_vals = np.array([e for _,e in per_zero_times])

# Fit power-law on log-log
mask = e_vals > 0
p_fit = np.polyfit(np.log(T_vals[mask]), np.log(e_vals[mask]), 1)
print(f'Empirical cost: cost = {np.exp(p_fit[1]):.3e} * T^{p_fit[0]:.3f}')

fig, axes = plt.subplots(2, 1, figsize=(7, 8))
ax = axes[0]
# Plot existing vs newly refined T-values on a number line
ax.scatter([float(x) for x in existing], np.zeros(1455), s=3, alpha=0.5, label='dps=80 prior (1455)', color='#1f77b4')
new_T = [float(b38[i]) for i in sorted(refined.keys())]
ax.scatter(new_T, np.ones(len(new_T))*0.5, s=8, alpha=0.7, label=f'dps=80 new ({len(refined)})', color='#d62728')
ax.scatter([float(b38[i]) for i in range(3405) if i not in refined and i >= 1455], 
 np.ones(3405-1455-len(refined))*1.0, s=3, alpha=0.3, label='still approx (dps=38)', color='gray')
ax.set_xlabel('Imaginary part T of L(Δ,s) zero')
ax.set_yticks([]); ax.set_xlim(0, 2300)
ax.set_title('A. L(Δ,s) zero refinement progress (cell budget 3600s)')
ax.legend(loc='upper right', fontsize=8)

ax = axes[1]
ax.scatter(T_vals, e_vals, s=8, alpha=0.5, label='per-zero refinement time')
Tg = np.linspace(50, 2200, 200)
ax.plot(Tg, np.exp(p_fit[1])*Tg**p_fit[0], 'r-', 
 label=f'fit: cost = {np.exp(p_fit[1]):.2e}·T^{p_fit[0]:.2f}')
ax.set_xlabel('T'); ax.set_ylabel('elapsed (s)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_title('B. Empirical cost per zero (dps=80 refinement)')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('ldelta_refinement_progress.png', dpi=120)
plt.close()
print('saved figure')


Refined 388 new dps=80 zeros in ~2642s wall (0.147/s aggregate)
Time to finish remaining 1562 zeros at current pace: 10636s = 3.0 hours
Empirical cost: cost = 6.717e-08 * T^2.679


saved figure
